In [ ]:
import os
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds, reproject
from rasterio.enums import Resampling
from affine import Affine
from shapely.geometry import Point, box, LineString
from shapely.ops import unary_union, linemerge
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
from matplotlib.collections import LineCollection
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from PIL import Image

# ============================================================
# 👉 全局字体与 PDF 导出设置
# ============================================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'Times New Roman'

mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12

# ============================================================
# 0. Paths (👉 已更新为最新的三级河流路径)
# ============================================================
dem_path      = r"E:\DEM30_Gebaini\DEM30.tif"
boundary_path = r"E:\矢量边界\gz_bountary\gz_city.shp"
nc_path       = r"E:\cn05PRE1961-2025\guanzhong_CN05.1_Pre_1961_2025_daily_025x025.nc"

river4_path   = r"E:\矢量边界\四级河流\River4_polyline.shp"
river3_path   = r"E:\矢量边界\中国三级及以上河流\R3\hyd2_4l.shp"  # <--- 新路径在这里

out_pdf       = r"E:\output\guanzhong_dem_precip_river.pdf"
temp_dem_png  = r"E:\output\temp_dem_base.png"  
transparent_bg = False

# ============================================================
# Helpers
# ============================================================
def format_lon(x, pos=None): return f"{int(x)}°E"
def format_lat(y, pos=None): return f"{int(y)}°N"

def inner_ticks(vmin, vmax, step):
    start = np.ceil(vmin / step) * step
    end = np.floor(vmax / step) * step
    if end < start: return []
    return np.arange(start, end + 0.1, step)

def build_dem_ticks_and_boundaries(dem_max, tick_start=100, target_ticks=7, sub_bins=2):
    dem_end = int(np.ceil(dem_max / 100.0) * 100)
    span = max(dem_end - tick_start, 100)
    raw_step = span / (target_ticks - 1)
    tick_step = int(np.ceil(raw_step / 100.0) * 100)
    ticks = np.arange(tick_start, dem_end + 1, tick_step, dtype=int)
    if ticks[-1] != dem_end: ticks = np.append(ticks, dem_end)
    boundaries = []
    for a, b in zip(ticks[:-1], ticks[1:]):
        boundaries.append(np.linspace(a, b, sub_bins + 1))
    boundaries = np.unique(np.concatenate(boundaries)).astype(float)
    return ticks, boundaries, tick_step

def true_clip_to_bbox(gdf_4326, bbox_expand):
    bbox_poly = box(*bbox_expand)
    fast = gdf_4326.cx[bbox_expand[0]:bbox_expand[2], bbox_expand[1]:bbox_expand[3]].copy()
    return gpd.clip(fast, bbox_poly)

def plot_trend_text(ax, text, line, bbox, fontsize=11, offset_pts=8, fig_w=10):
    if line is None or line.is_empty: return
    
    if line.geom_type == 'MultiLineString':
        line = max(line.geoms, key=lambda g: g.length)
    elif line.geom_type != 'LineString':
        return
        
    coords = list(line.coords)
    if coords[0][0] > coords[-1][0]:
        coords = coords[::-1]
        line = LineString(coords)
        
    x_range = bbox[2] - bbox[0]
    text_len_data = len(text) * (fontsize / 72.0) * (x_range / fig_w) * 0.45
    
    if line.length < text_len_data * 1.3:
        return
        
    mid_d = line.length / 2
    d1 = max(0, mid_d - text_len_data / 2)
    d2 = min(line.length, mid_d + text_len_data / 2)
    
    pt1 = line.interpolate(d1)
    pt2 = line.interpolate(d2)
    pt_mid = line.interpolate(mid_d)
    
    dx = pt2.x - pt1.x
    dy = pt2.y - pt1.y
    angle = np.degrees(np.arctan2(dy, dx))
    
    norm_len = np.hypot(dx, dy)
    if norm_len == 0: return
    nx, ny = -dy/norm_len, dx/norm_len
    
    if ny < 0: nx, ny = -nx, -ny 
        
    offset_data = (offset_pts / 72.0) * (x_range / fig_w)
    cx = pt_mid.x + nx * offset_data
    cy = pt_mid.y + ny * offset_data
    
    if not (bbox[0] + 0.05 < cx < bbox[2] - 0.05 and bbox[1] + 0.05 < cy < bbox[3] - 0.05):
        return
        
    ax.text(cx, cy, text, fontsize=fontsize, color='#003399',
            fontstyle='italic', fontweight='bold',
            rotation=angle, ha='center', va='center', zorder=4,
            path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])

# ============================================================
# 1. Boundary & Bbox
# ============================================================
gdf = gpd.read_file(boundary_path)
gdf_4326 = gdf.to_crs("EPSG:4326")
gdf_4326["geometry"] = gdf_4326.geometry.buffer(0)
gdf_outer = gdf_4326.dissolve()
poly_union = gdf_outer.geometry.union_all()

minx, miny, maxx, maxy = gdf_outer.total_bounds
expand = 0.5
bbox_expand = (minx-expand, miny-expand, maxx+expand, maxy+expand)

rings = []
geom = gdf_outer.geometry.iloc[0]
for poly in (geom.geoms if hasattr(geom, "geoms") else [geom]):
    x, y = poly.exterior.xy
    rings.append((np.array(x), np.array(y)))

# ============================================================
# 1b. Rivers (双数据源融合与暴力检索)
# ============================================================
# 读取四级河流
r4 = gpd.read_file(river4_path, encoding='gbk') 
if r4.crs is None: r4 = r4.set_crs("EPSG:4326", allow_override=True)
else: r4 = r4.to_crs("EPSG:4326")

# 读取三级以上河流
r3 = gpd.read_file(river3_path, encoding='gbk') 
if r3.crs is None: r3 = r3.set_crs("EPSG:4326", allow_override=True)
else: r3 = r3.to_crs("EPSG:4326")

# 拼接两个数据集
rivers_combined = gpd.GeoDataFrame(pd.concat([r4, r3], ignore_index=True), crs="EPSG:4326")
rivers_combined = rivers_combined[rivers_combined.geom_type.isin(["LineString", "MultiLineString"])].copy()

# 裁剪到研究区
rivers_in_bbox = true_clip_to_bbox(rivers_combined, bbox_expand)

# 提取所有线段用于绘制，保证 100% 不丢河！
river_segments = []
for geom in rivers_in_bbox.geometry:
    if geom.geom_type == 'LineString':
        river_segments.append(np.column_stack(geom.xy))
    elif geom.geom_type == 'MultiLineString':
        for line in geom.geoms:
            river_segments.append(np.column_stack(line.xy))

# 关中及周边核心水系字典
MATCH_DICT = {
    "渭河": "Weihe", "泾河": "Jinghe", "北洛河": "Beiluohe", "洛河": "Luohe",
    "千河": "Qianhe", "石川河": "Shichuanhe", "灞河": "Bahe", "浐河": "Chanhe",
    "沣河": "Fenghe", "涝河": "Laohe", "潏河": "Juehe", "滈河": "Haohe", "黑河": "Heihe",
    "丹江": "Danjiang", "汉江": "Hanjiang", "嘉陵江": "Jialingjiang", "黄河": "Huanghe",
    "唐白河": "Tangbai", "沙河": "Shahe", "白龙江": "Bailongjiang", "洮河": "Taohe",
    "伊洛河": "Yiluohe", "沁河": "Qinhe", "漳河": "Zhanghe", "无定河": "Wudinghe"
}

# 暴力全字段检索函数：不管列名叫什么，只要包含河流名就抓出来
def get_match_name(row):
    for val in row.values:
        if isinstance(val, str):
            for key in MATCH_DICT.keys():
                if key in val:
                    return key
    return None

rivers_in_bbox['match_key'] = rivers_in_bbox.apply(get_match_name, axis=1)

rivers_to_label = [] 
for r_key, group in rivers_in_bbox.groupby('match_key'):
    if not r_key: continue
    en_name = MATCH_DICT[r_key]
    
    union_geom = unary_union(group.geometry.values)
    if union_geom.geom_type == "LineString": 
        merged = union_geom
    elif union_geom.geom_type == "MultiLineString": 
        merged = linemerge(union_geom)
    else:
        lines = [g for g in union_geom.geoms if g.geom_type in ["LineString", "MultiLineString"]]
        merged = linemerge(lines) if lines else union_geom
        
    rivers_to_label.append((en_name, merged))

# ============================================================
# 2. DEM (PNG 预渲染技术，完美兼容 AI)
# ============================================================
deg_res_300m = 300.0 / 111320.0
with rasterio.open(dem_path) as src:
    dem_bounds = transform_bounds("EPSG:4326", src.crs, *bbox_expand, densify_pts=21) if src.crs.to_string() != "EPSG:4326" else bbox_expand
    window = from_bounds(*dem_bounds, transform=src.transform)
    
    resx, resy = src.res
    native_res = (abs(resx) + abs(resy)) / 2.0
    scale = (300.0 if src.crs.is_projected else deg_res_300m) / native_res
    
    out_h = max(int(window.height * scale), 1) if scale < 1 else int(window.height)
    out_w = max(int(window.width  * scale), 1) if scale < 1 else int(window.width)

    dem_small = src.read(1, window=window, out_shape=(out_h, out_w), resampling=Resampling.bilinear).astype("float32")
    if src.nodata is not None: dem_small[dem_small == src.nodata] = np.nan
    tr_small = src.window_transform(window) * Affine.scale(window.width/out_w, window.height/out_h)

dst_w, dst_h = max(int((bbox_expand[2] - bbox_expand[0]) / deg_res_300m), 1), max(int((bbox_expand[3] - bbox_expand[1]) / deg_res_300m), 1)
dst_tr = rasterio.transform.from_bounds(*bbox_expand, dst_w, dst_h)

dem = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
reproject(
    source=dem_small, destination=dem, 
    src_transform=tr_small, src_crs=src.crs, 
    dst_transform=dst_tr, dst_crs="EPSG:4326", 
    resampling=Resampling.bilinear,
    src_nodata=np.nan, dst_nodata=np.nan
)

dem_extent = (dst_tr.c, dst_tr.c + dst_tr.a * dem.shape[1], dst_tr.f + dst_tr.e * dem.shape[0], dst_tr.f)
dem_ticks, dem_boundaries, _ = build_dem_ticks_and_boundaries(float(np.nanmax(dem)), 100, 7, 2)

dem_cmap = mpl.colormaps.get_cmap("Greys").resampled(len(dem_boundaries) - 1)
dem_cmap.set_bad(color='white', alpha=1.0)
norm = mpl.colors.BoundaryNorm(dem_boundaries, dem_cmap.N)

dem_normalized = norm(dem)
dem_rgba = dem_cmap(dem_normalized, bytes=True) 
os.makedirs(os.path.dirname(temp_dem_png), exist_ok=True)
Image.fromarray(dem_rgba, mode='RGBA').save(temp_dem_png, dpi=(300, 300))

# ============================================================
# 3. Precip
# ============================================================
ds = xr.open_dataset(nc_path, engine="netcdf4")
pre_sel = ds["pre"].sel(time=slice("1991-01-01", "2020-12-31"))
pre_mam = pre_sel.sel(time=pre_sel["time"].dt.month.isin([3, 4, 5]))
pre_clim_total = pre_mam.groupby("time.year").sum("time", skipna=True).mean("year", skipna=True)

lon2d, lat2d = np.meshgrid(pre_clim_total.lon.values, pre_clim_total.lat.values)
pts = gpd.GeoDataFrame({"p": pre_clim_total.values.ravel()}, geometry=[Point(xy) for xy in zip(lon2d.ravel(), lat2d.ravel())], crs="EPSG:4326")
pts_in = pts[pts.within(poly_union)].copy()

px, py, pv = pts_in.geometry.x.to_numpy(), pts_in.geometry.y.to_numpy(), pts_in["p"].to_numpy()

# ============================================================
# 4. Plot
# ============================================================
fig, ax = plt.subplots(figsize=(10, 8), dpi=220)
fig.subplots_adjust(left=0.08, right=0.75, bottom=0.11, top=0.98)
ax.set(xlim=(bbox_expand[0], bbox_expand[2]), ylim=(bbox_expand[1], bbox_expand[3]), aspect="equal")
ax.margins(0)

# 贴入 DEM PNG
dem_img = plt.imread(temp_dem_png)
ax.imshow(dem_img, extent=dem_extent, origin="upper", zorder=0, interpolation='bilinear')

# 散点图
cmap_pre = mpl.colormaps.get_cmap("turbo_r")
norm_pre = mpl.colors.Normalize(vmin=80, vmax=170)
sc = ax.scatter(
    px, py, c=pv, s=22,
    cmap=cmap_pre, norm=norm_pre,
    edgecolors="none", alpha=1.0, zorder=2
)

# 绘制所有河流线（三级 + 四级全部画出）
if len(river_segments) > 0:
    ax.add_collection(LineCollection(river_segments, linewidths=2.6, alpha=0.95, zorder=2.55, color="white"))
    ax.add_collection(LineCollection(river_segments, linewidths=1.4, alpha=0.98, zorder=2.6, color="#1f77ff"))

# 绘制河流标注
for en_name, merged_line in rivers_to_label:
    plot_trend_text(ax, en_name, merged_line, bbox=bbox_expand, fontsize=11, offset_pts=8, fig_w=10)

# 边界线
for x, y in rings: ax.plot(x, y, color="black", linewidth=0.9, zorder=3)
ax.set_xticks(inner_ticks(bbox_expand[0], bbox_expand[2], 2))
ax.set_yticks(inner_ticks(bbox_expand[1], bbox_expand[3], 1))
ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
ax.yaxis.set_major_formatter(FuncFormatter(format_lat))

# ============================================================
# 5. Legends & Save
# ============================================================
dummy_im = ax.imshow(dem, extent=dem_extent, origin="upper", cmap=dem_cmap, norm=norm, visible=False)
cax_dem = fig.add_axes([0.12, 0.055, 0.60, 0.025])
cbar_dem = plt.colorbar(dummy_im, cax=cax_dem, orientation="horizontal", ticks=dem_ticks)
cbar_dem.set_label("Elevation (m)", fontsize=14)

legend_vals = [80, 100, 120, 140, 160]
handles = []
for v in legend_vals:
    handles.append(Line2D([0], [0], marker='o', color='w', 
                          markerfacecolor=cmap_pre(norm_pre(v)), 
                          markersize=10))

ax.legend(handles, [f"{v}" for v in legend_vals], 
          title="Precipitation\n(mm, Mar–May)", 
          loc='center left', bbox_to_anchor=(1.03, 0.5), 
          frameon=False, title_fontsize=13, fontsize=12)

fig.savefig(out_pdf, format="pdf", dpi=300, bbox_inches="tight", transparent=transparent_bg)
print(f"\n✅ 最终 PDF 已保存至: {out_pdf}")
plt.show()
